In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./dados/Base_de_Dados_Betfair Exchange_Filtrada_2025-07-03_PL.csv")

In [5]:
# Obter os times que aparecem nas colunas "Home" (mandantes) e "Away" (visitantes)
times_home = df["Home"].unique()
times_away = df["Away"].unique()

# Combinar os dois arrays e remover duplicatas
todos_os_times = pd.Series(list(times_home) + list(times_away)).drop_duplicates()

# Ordenar em ordem alfabética e resetar o índice
todos_os_times = todos_os_times.sort_values().reset_index(drop=True)

# Exibir a lista de times
print(todos_os_times)

0             Arsenal
1         Aston Villa
2         Bournemouth
3           Brentford
4            Brighton
5             Burnley
6             Chelsea
7      Crystal Palace
8             Everton
9              Fulham
10            Ipswich
11          Leicester
12          Liverpool
13              Luton
14    Manchester City
15     Manchester Utd
16          Newcastle
17         Nottingham
18      Sheffield Utd
19        Southampton
20          Tottenham
21           West Ham
22             Wolves
dtype: object


In [6]:
df = df[["Date", "League", "Home", "Away", "Goals_H_FT", "Goals_A_FT", "Goals_Min_H","Goals_Min_A", "Odd_H_FT", "Odd_D_FT", "Odd_A_FT", "Odd_Over25_FT", "Odd_BTTS_Yes"]]

,Date,League,Home,Away,Goals_H_FT,Goals_A_FT,Goals_Min_H,Goals_Min_A,Odd_H_FT,Odd_D_FT,Odd_A_FT,Odd_Over25_FT,Odd_BTTS_Yes
0,2024-03-16,ENGLAND 1,Burnley,Brentford,2,1,"[10, 62]",[83],3.35,3.75,2.26,1.81,1.70
1,2024-03-16,ENGLAND 1,Luton,Nottingham,1,1,[89],[34],2.88,3.75,2.56,1.71,1.60
2,2024-03-16,ENGLAND 1,Fulham,Tottenham,3,0,"[42, 49, 61]",[],3.50,4.10,2.10,1.55,1.53
3,2024-03-17,ENGLAND 1,West Ham,Aston Villa,1,1,[29],[79],2.82,3.85,2.56,1.60,1.52
4,2024-03-30,ENGLAND 1,Newcastle,West Ham,4,3,"[6, 77, 83, 90]","[21, 45, 48]",1.87,4.30,4.10,1.51,1.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,2025-05-25,ENGLAND 1,Newcastle,Everton,0,1,[],[65],1.31,6.00,12.50,1.61,2.04
463,2025-05-25,ENGLAND 1,Nottingham,Chelsea,0,1,[],[50],3.55,3.95,2.14,1.66,1.60
464,2025-05-25,ENGLAND 1,Southampton,Arsenal,1,2,[56],"[43, 89]",13.00,6.80,1.27,1.43,1.90
465,2025-05-25,ENGLAND 1,Tottenham,Brighton,1,4,[17],"[51, 64, 88, 90]",5.00,4.80,1.67,1.42,1.49


In [8]:
# Filtrar as partidas onde a odd do mandante ou do visitante é <= 1.80
filtro = (df["Odd_H_FT"] <= 1.80) | (df["Odd_A_FT"] <= 1.80)
partidas_filtradas = df[filtro]

In [9]:
partidas_filtradas

,Date,League,Home,Away,Goals_H_FT,Goals_A_FT,Goals_Min_H,Goals_Min_A,Odd_H_FT,Odd_D_FT,Odd_A_FT,Odd_Over25_FT,Odd_BTTS_Yes
6,2024-03-30,ENGLAND 1,Chelsea,Burnley,2,2,"[44, 78]","[47, 81]",1.32,6.4,11.00,1.53,1.94
8,2024-03-30,ENGLAND 1,Sheffield Utd,Fulham,3,3,"[58, 68, 70]","[62, 86, 90]",4.70,4.2,1.79,1.71,1.73
9,2024-03-30,ENGLAND 1,Tottenham,Luton,2,1,"[51, 86]",[3],1.24,8.2,13.50,1.27,1.66
10,2024-03-30,ENGLAND 1,Aston Villa,Wolves,2,0,"[36, 65]",[],1.64,4.5,5.90,1.65,1.73
12,2024-03-31,ENGLAND 1,Liverpool,Brighton,2,1,"[27, 65]",[2],1.38,6.2,8.40,1.34,1.59
...,...,...,...,...,...,...,...,...,...,...,...,...,...
460,2025-05-25,ENGLAND 1,Liverpool,Crystal Palace,1,1,[84],[9],1.43,6.2,6.80,1.33,1.53
461,2025-05-25,ENGLAND 1,Manchester Utd,Aston Villa,2,0,"[76, 87]",[],4.70,4.3,1.79,1.65,1.67
462,2025-05-25,ENGLAND 1,Newcastle,Everton,0,1,[],[65],1.31,6.0,12.50,1.61,2.04
464,2025-05-25,ENGLAND 1,Southampton,Arsenal,1,2,[56],"[43, 89]",13.00,6.8,1.27,1.43,1.90


In [10]:
# Filtrar jogos com odd do mandante ou visitante <= 1.80
filtro = (df["Odd_H_FT"] <= 1.80) | (df["Odd_A_FT"] <= 1.80)
df_filtrado = df[filtro].copy()

# Criar coluna com menor odd
df_filtrado["Menor_Odd"] = df_filtrado[["Odd_H_FT", "Odd_A_FT"]].min(axis=1)

# Verificar quem é o favorito
df_filtrado["Favorito"] = df_filtrado.apply(
    lambda row: "Home" if row["Odd_H_FT"] < row["Odd_A_FT"] else "Away",
    axis=1
)

# Verificar quem venceu
df_filtrado["Vencedor"] = df_filtrado.apply(
    lambda row: "Home" if row["Goals_H_FT"] > row["Goals_A_FT"]
    else ("Away" if row["Goals_A_FT"] > row["Goals_H_FT"] else "Empate"),
    axis=1
)

# Selecionar apenas as partidas em que o favorito venceu
vitorias_do_favorito = df_filtrado[df_filtrado["Favorito"] == df_filtrado["Vencedor"]]

In [11]:
vitorias_do_favorito

,Date,League,Home,Away,Goals_H_FT,Goals_A_FT,Goals_Min_H,Goals_Min_A,Odd_H_FT,Odd_D_FT,Odd_A_FT,Odd_Over25_FT,Odd_BTTS_Yes,Menor_Odd,Favorito,Vencedor
9,2024-03-30,ENGLAND 1,Tottenham,Luton,2,1,"[51, 86]",[3],1.24,8.2,13.50,1.27,1.66,1.24,Home,Home
10,2024-03-30,ENGLAND 1,Aston Villa,Wolves,2,0,"[36, 65]",[],1.64,4.5,5.90,1.65,1.73,1.64,Home,Home
12,2024-03-31,ENGLAND 1,Liverpool,Brighton,2,1,"[27, 65]",[2],1.38,6.2,8.40,1.34,1.59,1.38,Home,Home
19,2024-04-03,ENGLAND 1,Arsenal,Luton,2,0,"[24, 44]",[],1.12,13.0,27.00,1.28,2.18,1.12,Home,Home
21,2024-04-03,ENGLAND 1,Manchester City,Aston Villa,4,1,"[11, 45, 62, 69]",[20],1.29,7.0,11.50,1.45,1.83,1.29,Home,Home
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
456,2025-05-20,ENGLAND 1,Manchester City,Bournemouth,3,1,"[14, 38, 89]",[90],1.39,5.7,9.00,1.48,1.75,1.39,Home,Home
457,2025-05-25,ENGLAND 1,Bournemouth,Leicester,2,0,"[74, 88]",[],1.34,6.2,10.00,1.42,1.76,1.34,Home,Home
458,2025-05-25,ENGLAND 1,Fulham,Manchester City,0,2,[],"[21, 72]",5.70,4.6,1.64,1.58,1.66,1.64,Away,Away
464,2025-05-25,ENGLAND 1,Southampton,Arsenal,1,2,[56],"[43, 89]",13.00,6.8,1.27,1.43,1.90,1.27,Away,Away
